# 2) Linear Programming in Linopy — Exercises

**Goals:** compute dispatch costs by hand, write a merit-order price function, build a dispatch LP in Linopy for any number of plants, read shadow prices, and see where the LP beats the merit order (a CO₂ cap).

All tasks use the plants below. Units: MW for generation, €/MWh for costs and prices, € for total cost, t/MWh for emissions.

In [1]:
import math
import linopy
import pandas as pd

plants = pd.DataFrame(
    {"marginal_cost": [10, 30, 80], "p_max": [300, 800, 500], "co2_t_per_MWh": [0.0, 1.1, 0.4]},
    index=pd.Index(["nuclear", "lignite", "gas"], name="plant"),
)
plants

         marginal_cost  p_max  co2_t_per_MWh
plant                                       
nuclear             10    300            0.0
lignite             30    800            1.1
gas                 80    500            0.4

### Warm-ups

**1. Cost of a given dispatch**

Write a function that returns the total cost (€) of running each plant at the given output (MW) for one hour. This is the LP objective, computed by hand.

In [2]:
def dispatch_cost(plants, generation_MW):
    """generation_MW: dict plant -> MW. Return total cost in €."""
    ### BEGIN SOLUTION
    return float(sum(mw * plants.loc[name, "marginal_cost"] for name, mw in generation_MW.items()))
    ### END SOLUTION

In [3]:
assert math.isclose(dispatch_cost(plants, {"nuclear": 300, "lignite": 700, "gas": 0}), 24_000), \
    "300 MW x 10 €/MWh + 700 MW x 30 €/MWh"
assert math.isclose(dispatch_cost(plants, {"gas": 100}), 8_000)

**2. Merit-order price**

Return the clearing price (€/MWh) for one hour: sort plants by marginal cost and find the cost of the plant that covers the last MW of demand.

In [4]:
def merit_order_price(plants, demand_MW):
    """Return the marginal cost (€/MWh) of the plant that serves the last MW."""
    ### BEGIN SOLUTION
    order = plants.sort_values("marginal_cost")
    cumulative = order["p_max"].cumsum()
    return float(order.loc[cumulative >= demand_MW, "marginal_cost"].iloc[0])
    ### END SOLUTION

In [5]:
assert merit_order_price(plants, 200) == 10, "Nuclear alone covers 200 MW"
assert merit_order_price(plants, 1000) == 30, "Did you sort by marginal cost?"
assert merit_order_price(plants, 1200) == 80

### Core

**3. Dispatch LP for any set of plants**

Generalise the lecture's model: one variable per plant (0 to `p_max`), a constraint named `"demand"` so that supply equals demand, and minimise total cost. Return the solved model and the generation variable.

In [6]:
def build_dispatch_lp(plants, demand_MW):
    """Return (model, generation) after solving with HiGHS."""
    ### BEGIN SOLUTION
    m = linopy.Model()
    g = m.add_variables(lower=0, upper=plants["p_max"], coords=[plants.index], name="generation")
    m.add_constraints(g.sum() == demand_MW, name="demand")
    m.add_objective((g * plants["marginal_cost"]).sum())
    m.solve(solver_name="highs", output_flag=False)
    return m, g
    ### END SOLUTION

In [7]:
m, g = build_dispatch_lp(plants, 1000)
sol = g.solution.to_pandas()
assert math.isclose(m.objective.value, 24_000), "Expected 300 MW nuclear + 700 MW lignite"
assert math.isclose(sol["gas"], 0, abs_tol=1e-6)
sol

plant
nuclear    300.0
lignite    700.0
gas          0.0
Name: solution, dtype: float64

**4. Read the price from the solver**

Return the shadow price (dual value) of the `"demand"` constraint in €/MWh. Then compare it with your merit-order price for several demand levels.

In [8]:
def lp_price(model):
    """Return the shadow price of the demand constraint in €/MWh."""
    ### BEGIN SOLUTION
    return float(model.constraints["demand"].dual)
    ### END SOLUTION

In [9]:
for d in [200, 700, 1300]:
    m, _ = build_dispatch_lp(plants, d)
    assert math.isclose(lp_price(m), merit_order_price(plants, d)), f"Prices differ at {d} MW"
print("LP prices match the merit order at 200, 700 and 1300 MW.")

LP prices match the merit order at 200, 700 and 1300 MW.


**5. Interpretation**

At 1,000 MW of demand the price is 30 €/MWh, although nuclear runs at only 10 €/MWh. Why does the nuclear plant get paid 30 €/MWh, and why is that not a mistake?

<!-- BEGIN SOLUTION -->
The price is set by the **marginal** plant — the one that would serve one more MW. At 1,000 MW nuclear is already full, so the next MW comes from lignite at 30 €/MWh. Every plant is paid this clearing price; the difference (30 − 10 = 20 €/MWh for nuclear) is its *inframarginal rent*, which pays back its fixed costs.
<!-- END SOLUTION -->

### Challenge

**6. A CO₂ cap — where the merit order fails**

Demand is 1,000 MW and emissions may not exceed 600 t in this hour. Add this constraint to the LP (name it `"co2_cap"`) and return the solved model. Then look at the dispatch and the price.

*Think first:* without the cap, lignite runs at 700 MW (770 t). Is the cap binding? What does the cheapest *allowed* dispatch look like?

In [10]:
def dispatch_with_co2_cap(plants, demand_MW, cap_t):
    """Dispatch LP with an emissions cap (t). Return (model, generation)."""
    ### BEGIN SOLUTION
    m = linopy.Model()
    g = m.add_variables(lower=0, upper=plants["p_max"], coords=[plants.index], name="generation")
    m.add_constraints(g.sum() == demand_MW, name="demand")
    m.add_constraints((g * plants["co2_t_per_MWh"]).sum() <= cap_t, name="co2_cap")
    m.add_objective((g * plants["marginal_cost"]).sum())
    m.solve(solver_name="highs", output_flag=False)
    return m, g
    ### END SOLUTION

In [11]:
m, g = dispatch_with_co2_cap(plants, 1000, cap_t=600)
sol = g.solution.to_pandas()
# 300 MW nuclear; lignite L and gas G with L + G = 700 and 1.1 L + 0.4 G = 600  ->  L = 320/0.7 ≈ 457.1 MW
assert math.isclose(sol["lignite"], 320 / 0.7, rel_tol=1e-6), "Is the CO2 cap binding?"
assert math.isclose(m.objective.value, 3000 + 30 * 320 / 0.7 + 80 * (700 - 320 / 0.7), rel_tol=1e-6)
print(sol.round(1))
print(f"Price: {lp_price(m):.2f} €/MWh")

plant
nuclear    300.0
lignite    457.1
gas        242.9
Name: solution, dtype: float64
Price: 108.57 €/MWh


**7. Interpretation**

The price you just printed is **higher than the marginal cost of every plant**. Explain where it comes from. (Hint: what has to change if demand rises by 1 MW but emissions may not?)

<!-- BEGIN SOLUTION -->
With the cap binding, one extra MW cannot simply come from lignite or gas. Total output must rise by 1 MW while emissions stay at the cap, so lignite must go *down* and gas *up*: from 1.1·ΔL + 0.4·ΔG = 0 and ΔL + ΔG = 1 we get ΔL ≈ −0.571 MW and ΔG ≈ +1.571 MW. The cost change is 30·(−0.571) + 80·1.571 ≈ 108.57 €/MWh — the price of serving one more MW *and* keeping emissions under the cap. The merit order cannot find this, because no single plant is "marginal".
<!-- END SOLUTION -->

In [12]:
assert math.isclose(lp_price(m), 30 * (-0.4 / 0.7) + 80 * (1.1 / 0.7), rel_tol=1e-6)